[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SekyungHan/ai-power-textbook-labs/blob/master/solutions/ch04_llm_prompting.ipynb)

# Ch 4: LLM 활용 기초 — 프롬프팅 실습 (정답)

In [ ]:
# Ch04: LLM 활용 기초 — 프롬프팅 실습
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'anthropic'])
import os

try:
    sys.path.insert(0, '..')
    from utils.style import set_style
    set_style()
except:
    pass

## 1. API 설정 (Anthropic Claude)

In [ ]:
# Anthropic API 설정 (키가 없으면 pre-recorded 모드로 실행)
USE_API = False
try:
    import anthropic
    client = anthropic.Anthropic()
    # 간단한 테스트 호출
    test = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=10,
        messages=[{"role": "user", "content": "hi"}]
    )
    USE_API = True
    print("\u2713 API \uc5f0\uacb0 \uc131\uacf5")
except Exception as e:
    print(f"\u26a0 API \ud0a4 \uc5c6\uc74c \u2014 pre-recorded \ubaa8\ub4dc\ub85c \uc2e4\ud589\ud569\ub2c8\ub2e4")
    print(f"  (API \ud0a4\ub97c \uc124\uc815\ud558\ub824\uba74: os.environ['ANTHROPIC_API_KEY'] = 'your-key')")

def call_llm(prompt, system="", temperature=0.3, max_tokens=500):
    """API가 있으면 실제 호출, 없으면 None 반환"""
    if not USE_API:
        return None
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=max_tokens,
        temperature=temperature,
        system=system if system else anthropic.NOT_GIVEN,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

## 2. Demo 4.1: 프롬프트 버전별 Abstract 품질 비교

In [ ]:
research_info = """
주제: 딥러닝 기반 배전 계통 고장 위치 추정
방법: CNN + GRU 하이브리드 모델
데이터: 한국전력 실측 고장 데이터 (2019-2023)
결과: 기존 임피던스 방법 대비 위치 추정 오차 43% 감소
      고장 유형 분류 정확도 94.7%
"""

v1_prompt = f"다음 정보로 논문 abstract를 작성해주세요:\n{research_info}"

v2_prompt = f"""다음 정보로 IEEE 논문 abstract를 작성해주세요.
구조: (1) 배경/동기 (2) 목적 (3) 방법 (4) 주요 결과 (5) 기여
길이: 200단어 이내
톤: 학술적, 객관적
{research_info}"""

v3_prompt = f"""아래 예시와 같은 스타일로 abstract를 작성해주세요.

[예시]
This paper proposes a physics-informed neural network for
transient stability assessment of power systems. The proposed
method integrates swing equation constraints into the loss
function, achieving 97.2% accuracy on the IEEE 39-bus system
while requiring 60% less training data than conventional
deep learning approaches.

[작성할 내용]
{research_info}
구조: 배경-목적-방법-결과-기여, 200단어 이내"""

# Pre-recorded 응답
prerecorded = {
    1: "본 논문은 딥러닝 기반 배전 계통 고장 위치 추정 방법을 제안한다. CNN과 GRU 하이브리드 모델을 사용하여 한국전력 실측 데이터를 분석하였다. 기존 임피던스 방법 대비 오차가 43% 감소하였으며, 고장 유형 분류 정확도는 94.7%를 달성하였다.",
    2: "배경: 배전 계통의 고장 위치 추정은 신속한 복구를 위한 핵심 과제이다. 기존 임피던스 기반 방법은 분기선과 불균일 선로에서 정확도가 저하된다.\n\n목적: 본 논문은 딥러닝 하이브리드 모델을 활용하여 배전 계통 고장 위치 추정의 정확도를 향상시키는 방법을 제안한다.\n\n방법: CNN으로 고장 파형의 공간적 특징을 추출하고, GRU로 시계열 패턴을 학습하는 하이브리드 아키텍처를 설계하였다. 한국전력 실측 고장 데이터(2019-2023)를 사용하여 모델을 학습 및 검증하였다.\n\n결과: 제안 방법은 기존 임피던스 방법 대비 위치 추정 오차를 43% 감소시켰으며, 고장 유형 분류에서 94.7%의 정확도를 달성하였다.\n\n기여: 실측 데이터 기반 검증으로 실용적 적용 가능성을 입증하였다.",
    3: "Accurate fault location estimation in distribution networks is critical for rapid service restoration, yet conventional impedance-based methods suffer from degraded accuracy in networks with lateral branches and non-uniform conductors. This paper proposes a hybrid deep learning framework combining Convolutional Neural Networks (CNN) and Gated Recurrent Units (GRU) for distribution system fault location and classification. The CNN module extracts spatial features from fault waveforms while the GRU captures temporal dependencies. The model is trained and validated on field fault data from Korea Electric Power Corporation spanning 2019-2023. Experimental results demonstrate that the proposed method reduces fault location estimation error by 43% compared to the conventional impedance method, while achieving a fault type classification accuracy of 94.7%. These findings confirm the practical applicability of deep learning approaches to real-world distribution fault diagnosis."
}

for i, prompt in enumerate([v1_prompt, v2_prompt, v3_prompt], 1):
    result = call_llm(prompt, temperature=0.3)
    if result is None:
        result = prerecorded[i]
    print(f"=== Version {i} {'(단순 요청)' if i==1 else '(구조화)' if i==2 else '(Few-shot)'} ===")
    print(result)
    print()

## 3. Demo 4.2: MATPOWER 데이터 해석

In [ ]:
matpower_data = """
%% generator data
% bus  Pg  Qg  Qmax  Qmin  Vg  mBase  status  Pmax  Pmin
  1   232.4  -17.0  10.0  0.0  1.060  100  1  332.4  0
  2    40.0   42.4  50.0  -40  1.045  100  1  140.0  0
  3     0.0   23.4  40.0   0   1.010  100  1  100.0  0
  6     0.0   12.2  24.0  -6   1.070  100  1  100.0  0
  8     0.0   17.4  24.0  -6   1.090  100  1  100.0  0
"""

prompt = f"""다음은 MATPOWER case14의 발전기 데이터입니다:
{matpower_data}
작업:
1. 각 열의 의미를 설명하세요
2. 슬랙 버스(slack bus)를 식별하세요
3. 무효전력 범위가 가장 넓은 발전기는?
4. 결과를 정리된 표로 출력하세요
"""

prerecorded_response = """
## MATPOWER case14 발전기 데이터 분석

### 1. 열 설명
| 열명 | 의미 | 단위 |
|------|------|------|
| bus | 발전기 연결 모선 번호 | - |
| Pg | 유효전력 출력 | MW |
| Qg | 무효전력 출력 | Mvar |
| Qmax | 무효전력 상한 | Mvar |
| Qmin | 무효전력 하한 | Mvar |
| Vg | 전압 설정값 | p.u. |
| mBase | 기준 용량 | MVA |
| status | 운전 상태 (1=운전중) | - |
| Pmax | 유효전력 상한 | MW |
| Pmin | 유효전력 하한 | MW |

### 2. 슬랙 버스
Bus 1 — 유효전력 출력이 232.4 MW로 가장 높고, Pmax=332.4 MW로 출력 범위가 가장 넓습니다.

### 3. 무효전력 범위
| Bus | Qmax - Qmin | 범위 |
|-----|-------------|------|
| 1 | 10 - 0 | 10 Mvar |
| 2 | 50 - (-40) | **90 Mvar** |
| 3 | 40 - 0 | 40 Mvar |
| 6 | 24 - (-6) | 30 Mvar |
| 8 | 24 - (-6) | 30 Mvar |

→ **Bus 2**의 무효전력 범위가 90 Mvar로 가장 넓습니다.
"""

result = call_llm(prompt, system="전력 계통 전문가. IEEE 표준 용어 사용.", temperature=0)
if result is None:
    result = prerecorded_response
print(result)